**assignment 3 - NLPROC-ALV-B**

**Anja Kieburg, Friedemann Decker**

## Task 2 Programming

### Task 2 TIGER Corpus

Korpus TIGER 2.2.doc (April 2018)
from: https://www.ims.uni-stuttgart.de/en/research/resources/corpora/tiger/

TIGER Corpus
The TIGER Corpus consists of approximate 900,000 tokens (50,000 sentences) of German newspaper text, taken from the Frankfurter Rundschau. The corpus was semi-automatically POS-tagged and annotated with syntactic structure. Moreover, it contains morphological and lemma information for terminal nodes

In [2]:
# Setup AI
%load_ext jupyter_ai_magics
%load_ext dotenv
%dotenv

In [3]:
import pandas as pd
import numpy as np

**Step 1**: Import TIGER XML into Pandas, using only the entries under `s=Sätze`

In [4]:
tiger_corpus = pd.read_xml("tiger_release_aug07.corrected.16012013.xml",
                  xpath="//s/graph/terminals/t",
                  elems_only=False,
                  encoding="ISO-8859-1")

In [5]:
len_tiger_corpus = len(tiger_corpus)
len(tiger_corpus)

888238

In [6]:
tiger_corpus.head()

,id,word,lemma,pos,morph,case,number,gender,person,degree,tense,mood,secedge
0,s1_1,``,--,$(,--,--,--,--,--,--,--,--,NaN
1,s1_2,Ross,Ross,NE,Nom.Sg.Masc,Nom,Sg,Masc,--,--,--,--,NaN
2,s1_3,Perot,Perot,NE,Nom.Sg.Masc,Nom,Sg,Masc,--,--,--,--,NaN
3,s1_4,wäre,sein,VAFIN,3.Sg.Past.Subj,--,Sg,--,3,--,Past,Subj,NaN
4,s1_5,vielleicht,vielleicht,ADV,--,--,--,--,--,--,--,--,NaN


### Task 2 A) tag t that maximizes p(t|w)

#### Step 2: Required fields: Import "word" and "pos" into the DataFrame

In [7]:
words = tiger_corpus[["word", "pos"]]
words.head()

,word,pos
0,``,$(
1,Ross,NE
2,Perot,NE
3,wäre,VAFIN
4,vielleicht,ADV


#### Step 3: Count the number of unique combinations of "word" AND "pos" = frequency of occurrence in the corpus

In [8]:
word_pos_freq = words[["word", "pos"]].value_counts()
# Set -> df
words = word_pos_freq.reset_index(name="pos_freq")
words.head()

,word,pos,pos_freq
0,",","$,",43656
1,.,$.,41057
2,der,ART,25286
3,die,ART,21209
4,und,KON,16281


#### Step 4: Convert "word" to lowercase so that different parts of speech (POS) can be grouped into a "word_lower" category

In [9]:
word_lower = words["word"].str.lower()
words.insert(1, "word_lower", word_lower)
words.head()

,word,word_lower,pos,pos_freq
0,",",",","$,",43656
1,.,.,$.,41057
2,der,der,ART,25286
3,die,die,ART,21209
4,und,und,KON,16281


#### Step 5: Determine the number of words per "word_lower" category and enter it as "word_freq" in each row to calculate p(t|w).

In [10]:
word_freq = words.groupby("word_lower")["pos_freq"].transform("sum")
words.insert(4, "word_freq", word_freq)
words.head()

,word,word_lower,pos,pos_freq,word_freq
0,",",",","$,",43656,43659
1,.,.,$.,41057,41057
2,der,der,ART,25286,29662
3,die,die,ART,21209,30024
4,und,und,KON,16281,16786


#### Step 6: Calculate p_pos = p(t|w)

In [11]:
p_pos = words["pos_freq"] / words["word_freq"]
words.insert(5, "p_pos", p_pos)
words.head()

,word,word_lower,pos,pos_freq,word_freq,p_pos
0,",",",","$,",43656,43659,0.999931
1,.,.,$.,41057,41057,1.000000
2,der,der,ART,25286,29662,0.852471
3,die,die,ART,21209,30024,0.706402
4,und,und,KON,16281,16786,0.969915


#### Step 7: Keep only p_pos_max values in the DataFrame

In [12]:
words["max_p_pos"] = words.groupby("word_lower")["p_pos"].transform("max")
words = words[words["p_pos"] == words["max_p_pos"]]

words.head()

,word,word_lower,pos,pos_freq,word_freq,p_pos,max_p_pos
0,",",",","$,",43656,43659,0.999931,0.999931
1,.,.,$.,41057,41057,1.000000,1.000000
2,der,der,ART,25286,29662,0.852471,0.852471
3,die,die,ART,21209,30024,0.706402,0.706402
4,und,und,KON,16281,16786,0.969915,0.969915


#### Step 8: Delete columns that aren't needed (clean up). Rename the columns according to the assignment.

In [13]:
words = words.drop(columns = ["word_lower", "p_pos"])


In [14]:
words = words.rename(columns={'word': 'w', 'pos': 't', 'max_p_pos': 'p'})

**Result**: Computation of the tag **t** for each word **w** that maximizes **p(t|w)**.
List of the first 30 values:

In [15]:
words.head(30)

,w,t,pos_freq,word_freq,p
0,",","$,",43656,43659,0.999931
1,.,$.,41057,41057,1.000000
2,der,ART,25286,29662,0.852471
3,die,ART,21209,30024,0.706402
4,und,KON,16281,16786,0.969915
5,in,APPR,13711,15102,0.907893
6,den,ART,9486,9955,0.952888
7,'',$(,8811,8811,1.000000
8,``,$(,8809,8809,1.000000
9,von,APPR,8182,9033,0.905790


#### Appendix: Meaning of the Abbreviations Used for Parts of Speech (POS)

In [16]:
tiger_corpus_pos_abr = pd.read_xml("tiger_release_aug07.corrected.16012013.xml",
                  xpath="//head/annotation/feature/value",
                  elems_only=False,
                  encoding="ISO-8859-1")

In [17]:
tiger_corpus_pos_abr.head(56)

,name,value
0,$(,Sonstige Satzzeichen; satzintern
1,"$,",Komma
2,$.,Satzbeendende Interpunktion
3,--,<Nicht zugeordnet>
4,ADJA,Attributives Adjektiv
5,ADJD,Adverbiales oder prädikatives Adjektiv
6,ADV,Adverb
7,APPO,Postposition
8,APPR,Präposition; Zirkumposition links
9,APPRART,Präposition mit Artikel


### Task 2 B) - HMM tagger

#### Step 1) Split the corpus into a training set and test set.

In [18]:
#%%ai gemini:gemini-3.8-flash
#kurz, knapp: Welche Splitting-Ratio empfiehlst du für einen NLP-Datensatz mit 900.000 Einträgen?

In [19]:
# SOLL Anzahl Worte
len_train = int(len_tiger_corpus * 0.95)
len_test = len_tiger_corpus - len_train
print(f"Words in train set calc: {len_train}")
print(f"Words in test set calc: {len_test}")

Words in train set calc: 843826
Words in test set calc: 44412


In [20]:
#IST Anzahl Worte
train = tiger_corpus.iloc[:len_train]
train = train[["word", "pos"]]
test = tiger_corpus.iloc[len_train:]
test = test[["word", "pos"]]
print(f"Words in train set act: {len(train)}")
print(f"Words in test set act: {len(test)}")

Words in train set act: 843826
Words in test set act: 44412


#### Step 2) Counting of POS tags, POS tag - POS tag follower, word - POS tag combination

From the training set, count how often each (i) POS tag occurs,
how often one (ii) POS tag follows another and how often each (iii) word occurs with each POS tag.

In [31]:
# Step 2 (i) Occurence of POS tags
train_pos_count = train[["pos"]].value_counts()
# Set -> df
train_pos_count = train_pos_count.reset_index(name="pos_count")
train_pos_count.head()

,pos,pos_count
0,NN,175113
1,ART,92486
2,APPR,71802
3,ADJA,51747
4,NE,49643


In [22]:
# Step 2 (ii) POS tag follows another
# Testset mit Follower bauen
# Follower Spalte bauen, alle Zeilen um 1 Zeile nach oben, letzte Zeile = "NaN"
train_foll = pd.DataFrame()
train_foll[["word_foll", "pos_foll"]] = train[["word", "pos"]]
train_foll = train_foll.iloc[1:].reset_index(drop=True)
train_foll.loc[len(train_foll)] = np.nan

# Follower Spalte zu Testset hinzufügen
train = pd.concat([train, train_foll], axis=1)

In [23]:
train

,word,pos,word_foll,pos_foll
0,``,$(,Ross,NE
1,Ross,NE,Perot,NE
2,Perot,NE,wäre,VAFIN
3,wäre,VAFIN,vielleicht,ADV
4,vielleicht,ADV,ein,ART
...,...,...,...,...
843821,die,ART,grüne,ADJA
843822,grüne,ADJA,Lunge,NN
843823,Lunge,NN,der,ART
843824,der,ART,Zehn-Millionen-City,NN


In [24]:
# %%ai gemini:gemini-3.8-flash
# Mit welcher Pivot-Pandas/Numpy Funktion(en) formt man training_set_pos_pos_fol um, sodass "pos" die Zeilenbezeichnung,
# "pos_foll" die Spaltenbezeichnung und die Zellinhalte die Werte aus value_counts enthält?:
# training_set_pos_pos_foll_count = training_set[["pos", "pos_foll"]].value_counts()

In [25]:
train_pos_pos_foll_count = pd.crosstab(train["pos"], train["pos_foll"])
# train_pos_pos_foll_count

In [33]:
# Step 2 (iii) Occurence of word with each POS tag
train_word_pos_count = train[["word", "pos"]].value_counts()
# Set -> df
train_word_pos_count = train_word_pos_count.reset_index(name="word_pos_count")
# Gesamtanzahl pro POS tag zu jeder Kombination word_pos in neuer Spalte ergänzen
train_word_pos_count = train_word_pos_count.merge(train_pos_count[["pos", "pos_count"]], on="pos", how="left")
train_word_pos_count

,word,pos,word_pos_count,pos_count
0,",","$,",41320,41320
1,.,$.,39016,43233
2,der,ART,24084,92486
3,die,ART,20127,92486
4,und,KON,15366,20832
...,...,...,...,...
90501,erfrischt,VVPP,1,16790
90502,durchatmen,VVINF,1,12757
90503,Großstadtdschungel,NN,1,175113
90504,Metro-Manila,NE,1,49643


#### Step 3 Compute the (i) transition and (ii) observation probabilities of the HMM tagger from this.

In [27]:
# Step 3(i) transition probabilities
# Zeilensummen von trai_pos_pos_foll_count ermitteln, train_pos_foll_sum ist ein Set
train_pos_foll_sum = train_pos_pos_foll_count.sum(axis=1)
# Jeden Spaltenwert durch Zeilensumme teilen
train_pos_trans_prob = train_pos_pos_foll_count.div(train_pos_foll_sum, axis=0)
# train_pos_trans_prob

In [35]:
# Step 3(ii) observation probabilities (Gibt an, mit welcher Wahrscheinlichkeit ein Wort als bestimmte Wortart (POS) erscheint.)
train_word_pos_count["word_pos_prob"] = train_word_pos_count["word_pos_count"] / train_word_pos_count["pos_count"]
train_word_pos_count

,word,pos,word_pos_count,pos_count,word_pos_prob
0,",","$,",41320,41320,1.000000
1,.,$.,39016,43233,0.902459
2,der,ART,24084,92486,0.260407
3,die,ART,20127,92486,0.217622
4,und,KON,15366,20832,0.737615
...,...,...,...,...,...
90501,erfrischt,VVPP,1,16790,0.000060
90502,durchatmen,VVINF,1,12757,0.000078
90503,Großstadtdschungel,NN,1,175113,0.000006
90504,Metro-Manila,NE,1,49643,0.000020


d) Then implement the Viterbi algorithm so you can decode a test sentence.

e) Create a Viterbi table
that stores, for each position in the sentence and each possible tag, the probability of the most
probable path ending in that tag.

f) Initialize the table for the first word using the initial tag probabilities and the observation probability of that word.

g) Process the remaining words from left to right.

h) For each word, update the Viterbi table by considering all possible tags and the probabilities of the most probable paths leading to them.

i) Store both the best score and the preceding tag that achieved it (the backpointer).

j) After filling the table, backtrack from the last word to find the most probable sequence of tags.

k) Now run your algorithm on the test set. Report its error rate and compare its performance
to the most frequent tag baseline.